# 🤖 Prédiction de Prix Immobilier — Model Training

**Objectif** : Entraîner et comparer plusieurs modèles de régression pour prédire le prix des biens immobiliers marocains.

**Modèles testés** :
1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. Random Forest
5. Gradient Boosting
6. XGBoost

---

## 1. 📦 Setup & Imports

In [ ]:
import os, sys, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error
)
from sklearn.model_selection import cross_val_score

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('⚠️ xgboost non installé — pip install xgboost')

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print('✅ Imports OK')

## 2. 📂 Charger les Données ML

In [ ]:
final_dir = os.path.join('..', 'data', 'final')

X_train = pd.read_csv(os.path.join(final_dir, 'X_train.csv'))
X_test  = pd.read_csv(os.path.join(final_dir, 'X_test.csv'))
y_train = pd.read_csv(os.path.join(final_dir, 'y_train.csv'))['prix']
y_test  = pd.read_csv(os.path.join(final_dir, 'y_test.csv'))['prix']

print(f'📊 X_train : {X_train.shape}')
print(f'📊 X_test  : {X_test.shape}')
print(f'📊 y_train : {y_train.shape} — mean={y_train.mean():,.0f} MAD')
print(f'📊 y_test  : {y_test.shape}  — mean={y_test.mean():,.0f} MAD')
print(f'\n📋 Features : {list(X_train.columns)}')

## 3. 🏗️ Définition des Modèles

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (α=1.0)':     Ridge(alpha=1.0),
    'Lasso (α=100)':     Lasso(alpha=100),
    'Random Forest':     RandomForestRegressor(
                            n_estimators=200, max_depth=10,
                            min_samples_split=3, random_state=42
                         ),
    'Gradient Boosting':  GradientBoostingRegressor(
                            n_estimators=200, max_depth=4,
                            learning_rate=0.1, random_state=42
                         ),
}

if HAS_XGB:
    models['XGBoost'] = XGBRegressor(
        n_estimators=200, max_depth=4,
        learning_rate=0.1, random_state=42,
        verbosity=0
    )

print(f'🤖 {len(models)} modèles à entraîner :')
for name in models:
    print(f'  → {name}')

## 4. 🚀 Entraînement & Évaluation

In [ ]:
results = []

for name, model in models.items():
    print(f'\n{"─"*55}')
    print(f'🏋️ {name}')
    print(f'{"─"*55}')
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)
    
    # Metrics
    mae_train  = mean_absolute_error(y_train, y_pred_train)
    mae_test   = mean_absolute_error(y_test, y_pred_test)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    rmse_test  = np.sqrt(mean_squared_error(y_test, y_pred_test))
    r2_train   = r2_score(y_train, y_pred_train)
    r2_test    = r2_score(y_test, y_pred_test)
    mape_test  = mean_absolute_percentage_error(y_test, y_pred_test) * 100
    
    # Cross-validation (5-fold on train)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    
    print(f'  Train │ R²={r2_train:.4f} │ MAE={mae_train:,.0f} │ RMSE={rmse_train:,.0f}')
    print(f'  Test  │ R²={r2_test:.4f}  │ MAE={mae_test:,.0f}  │ RMSE={rmse_test:,.0f}')
    print(f'  MAPE  │ {mape_test:.1f}%')
    print(f'  CV(5) │ R² = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    
    overfit = r2_train - r2_test
    if overfit > 0.2:
        print(f'  ⚠️  Overfitting détecté (gap R² = {overfit:.3f})')
    
    results.append({
        'Modèle':      name,
        'R² Train':    round(r2_train, 4),
        'R² Test':     round(r2_test, 4),
        'MAE (MAD)':   round(mae_test),
        'RMSE (MAD)':  round(rmse_test),
        'MAPE (%)':    round(mape_test, 1),
        'CV R² (mean)': round(cv_scores.mean(), 4),
        'CV R² (std)':  round(cv_scores.std(), 4),
    })

print(f'\n{"═"*55}')
print('✅ Tous les modèles entraînés!')
print(f'{"═"*55}')

## 5. 📊 Tableau Comparatif

In [ ]:
df_results = pd.DataFrame(results).sort_values('R² Test', ascending=False)
df_results = df_results.reset_index(drop=True)

# Highlight le meilleur modèle
best_idx  = df_results['R² Test'].idxmax()
best_name = df_results.loc[best_idx, 'Modèle']
best_r2   = df_results.loc[best_idx, 'R² Test']
best_mae  = df_results.loc[best_idx, 'MAE (MAD)']

print(f'🏆 Meilleur modèle : {best_name} (R²={best_r2:.4f}, MAE={best_mae:,.0f} MAD)\n')

df_results.style.background_gradient(cmap='RdYlGn', subset=['R² Test', 'CV R² (mean)']) \
               .background_gradient(cmap='RdYlGn_r', subset=['MAE (MAD)', 'RMSE (MAD)', 'MAPE (%)'])

## 6. 📈 Visualisation des Performances

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

colors = sns.color_palette('viridis', len(df_results))

# R² Score
bars1 = axes[0].barh(df_results['Modèle'], df_results['R² Test'], color=colors)
axes[0].set_title('R² Score (Test)', fontsize=14, fontweight='bold')
axes[0].set_xlim(min(0, df_results['R² Test'].min() - 0.1), 1.05)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
for bar, val in zip(bars1, df_results['R² Test']):
    axes[0].text(max(val, 0) + 0.02, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontweight='bold')

# MAE
bars2 = axes[1].barh(df_results['Modèle'], df_results['MAE (MAD)'], color=colors)
axes[1].set_title('MAE — Erreur Moyenne (MAD)', fontsize=14, fontweight='bold')
for bar, val in zip(bars2, df_results['MAE (MAD)']):
    axes[1].text(val + 5000, bar.get_y() + bar.get_height()/2,
                f'{val:,.0f}', va='center', fontweight='bold')

# MAPE
bars3 = axes[2].barh(df_results['Modèle'], df_results['MAPE (%)'], color=colors)
axes[2].set_title('MAPE — Erreur Relative (%)', fontsize=14, fontweight='bold')
for bar, val in zip(bars3, df_results['MAPE (%)']):
    axes[2].text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontweight='bold')

plt.suptitle('Comparaison des Modèles de Prédiction de Prix', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# R² Train vs Test — détection overfitting
fig, ax = plt.subplots(figsize=(12, 5))

x = range(len(df_results))
width = 0.35

bars_train = ax.bar([i - width/2 for i in x], df_results['R² Train'], width,
                    label='R² Train', color='#3498db', alpha=0.8)
bars_test  = ax.bar([i + width/2 for i in x], df_results['R² Test'], width,
                    label='R² Test', color='#e74c3c', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(df_results['Modèle'], rotation=15, ha='right')
ax.set_ylabel('R² Score')
ax.set_title('R² Train vs Test — Détection d\'Overfitting', fontsize=14, fontweight='bold')
ax.legend()
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 7. 🔍 Feature Importance (Meilleur Modèle)

In [ ]:
# Identifier le meilleur modèle basé sur les arbres pour feature importance
tree_models = {n: m for n, m in models.items()
               if hasattr(m, 'feature_importances_')}

if tree_models:
    # Prendre le meilleur modèle tree-based
    tree_results = df_results[df_results['Modèle'].isin(tree_models.keys())]
    best_tree_name = tree_results.loc[tree_results['R² Test'].idxmax(), 'Modèle']
    best_tree = tree_models[best_tree_name]
    
    importances = best_tree.feature_importances_
    feat_imp = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': importances
    }).sort_values('Importance', ascending=True)
    
    fig, ax = plt.subplots(figsize=(12, 7))
    colors_imp = sns.color_palette('YlOrRd', len(feat_imp))
    ax.barh(feat_imp['Feature'], feat_imp['Importance'], color=colors_imp)
    ax.set_title(f'Feature Importance — {best_tree_name}', fontsize=15, fontweight='bold')
    ax.set_xlabel('Importance')
    
    for i, (idx, row) in enumerate(feat_imp.iterrows()):
        ax.text(row['Importance'] + 0.005, i, f'{row["Importance"]:.3f}',
                va='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # Top 5
    print(f'\n🔑 Top 5 features les plus importantes ({best_tree_name}) :')
    for _, row in feat_imp.tail(5).iloc[::-1].iterrows():
        print(f'  {row["Feature"]:>20s} │ {row["Importance"]:.4f} ({row["Importance"]*100:.1f}%)')
else:
    print('⚠️ Aucun modèle tree-based entraîné')

## 8. 📉 Prédictions vs Réalité

In [ ]:
# Scatter plot pour le meilleur modèle
best_model = models[best_name]
y_pred_best = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter : Réel vs Prédit
axes[0].scatter(y_test, y_pred_best, c='#3498db', alpha=0.7, s=80, edgecolors='white')
lims = [min(y_test.min(), min(y_pred_best)), max(y_test.max(), max(y_pred_best))]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Prédiction parfaite')
axes[0].set_xlabel('Prix Réel (MAD)', fontsize=12)
axes[0].set_ylabel('Prix Prédit (MAD)', fontsize=12)
axes[0].set_title(f'Réel vs Prédit — {best_name}', fontsize=14, fontweight='bold')
axes[0].legend()

# Résidus
residuals = y_test - y_pred_best
axes[1].scatter(y_pred_best, residuals, c='#e74c3c', alpha=0.7, s=80, edgecolors='white')
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Prix Prédit (MAD)', fontsize=12)
axes[1].set_ylabel('Résidu (MAD)', fontsize=12)
axes[1].set_title(f'Résidus — {best_name}', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Distribution des erreurs
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(residuals, bins=15, color='#9b59b6', edgecolor='white', alpha=0.85)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_title(f'Distribution des Résidus — {best_name}', fontsize=14, fontweight='bold')
ax.set_xlabel('Erreur (MAD)')
ax.set_ylabel('Fréquence')
print(f'\nMoyenne résidus : {residuals.mean():,.0f} MAD')
print(f'Std résidus     : {residuals.std():,.0f} MAD')
plt.tight_layout()
plt.show()

## 9. 📋 Détail des Prédictions (Test Set)

In [ ]:
df_pred = X_test.copy()
df_pred['Prix Réel (MAD)']  = y_test.values
df_pred['Prix Prédit (MAD)'] = y_pred_best.round(0)
df_pred['Erreur (MAD)']      = (y_test.values - y_pred_best).round(0)
df_pred['Erreur (%)']        = ((y_test.values - y_pred_best) / y_test.values * 100).round(1)

cols_show = ['Prix Réel (MAD)', 'Prix Prédit (MAD)', 'Erreur (MAD)', 'Erreur (%)']
df_pred[cols_show].style.background_gradient(
    cmap='RdYlGn_r', subset=['Erreur (MAD)'], vmin=-500000, vmax=500000
).format({
    'Prix Réel (MAD)':   '{:,.0f}',
    'Prix Prédit (MAD)': '{:,.0f}',
    'Erreur (MAD)':      '{:+,.0f}',
    'Erreur (%)':        '{:+.1f}%'
})

## 10. 💾 Sauvegarder le Meilleur Modèle

In [ ]:
# Sauvegarder le meilleur modèle
model_path = os.path.join(final_dir, 'best_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)

# Sauvegarder les résultats
results_path = os.path.join(final_dir, 'model_results.json')
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump({
        'best_model':   best_name,
        'r2_test':      float(best_r2),
        'mae_test':     float(best_mae),
        'features':     list(X_train.columns),
        'all_results':  results
    }, f, indent=2, ensure_ascii=False)

print(f'💾 Modèle sauvegardé : {model_path}')
print(f'📄 Résultats :         {results_path}')
print(f'\n🏆 {best_name} — R²={best_r2:.4f} │ MAE={best_mae:,.0f} MAD')

## 11. 🎯 Exemple de Prédiction

In [ ]:
# Charger scaler + encoders pour prédire sur de nouvelles données
with open(os.path.join(final_dir, 'scaler.pkl'), 'rb') as f:
    scaler = pickle.load(f)
with open(os.path.join(final_dir, 'encoders.pkl'), 'rb') as f:
    encoders = pickle.load(f)

print('Encodeurs disponibles :')
for col, enc in encoders.items():
    print(f'  {col} : {list(enc.classes_)}')

print(f'\nColonnes scaled : {list(X_train.columns)}')
print(f'\n🎯 Pour prédire un nouveau bien :')
print('   1. Encoder les catégorielles avec les encoders ci-dessus')
print('   2. Scaler les numériques avec le scaler sauvegardé')
print('   3. Appeler best_model.predict(X_new)')

## 12. 📊 Résumé Final

In [ ]:
print('═' * 60)
print('📊 RÉSUMÉ — MODÈLES DE PRÉDICTION DE PRIX')
print('═' * 60)
print(f'\n  Dataset   : {len(X_train)+len(X_test)} lignes × {X_train.shape[1]} features')
print(f'  Split     : {len(X_train)} train / {len(X_test)} test')
print(f'  Modèles   : {len(models)} testés')
print(f'\n  🏆 MEILLEUR : {best_name}')
print(f'     R²   = {best_r2:.4f}')
print(f'     MAE  = {best_mae:,.0f} MAD')
print(f'\n  Fichiers sauvegardés :')
print(f'     📦 {model_path}')
print(f'     📄 {results_path}')
print(f'\n  ⚠️  Note : Avec seulement {len(X_train)+len(X_test)} lignes, les modèles')
print(f'     sont indicatifs. Scraper plus de données pour de')
print(f'     meilleures performances.')
print('═' * 60)